# Clasificador de residuos en Google Colab
El cuaderno reproduce la CNN de diez categorías y su ajuste con TACO. Coloca el repositorio actualizado en Drive y agrega las imágenes internas indicadas en el README. Si la carpeta es compartida, añade un acceso directo en Mi unidad y verifica que puedas guardar en ella.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Preparar la ejecución
Se conserva el manifiesto versionado y su lista de exclusiones. La ejecución copia el checkpoint inicial como referencia para la comparación. El entrenamiento de ObjectCNN parte de una inicialización aleatoria y la etapa TACO parte del nuevo checkpoint de objetos. Las fotos de TACO se copian desde Drive si están disponibles o se descargan según el manifiesto.


In [ ]:
import importlib.util
import json
from pathlib import Path
import shutil
import subprocess
import sys

SOURCE = Path('/content/drive/MyDrive/clasificadorDeBasura')  # Ajusta la ruta a tu copia.
spec = importlib.util.spec_from_file_location('drive_workflow', SOURCE/'src/drive_workflow.py')
if not (SOURCE/'src/drive_workflow.py').is_file():
    raise FileNotFoundError('Ajusta SOURCE a la carpeta que contiene src/drive_workflow.py.')
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
PROJECT = workflow.prepare_run(SOURCE, Path('/content/residuos_ejecuciones'))

def run(script, *args):
    subprocess.run([sys.executable, script, *args], cwd=PROJECT, check=True)

def guardar_avance():
    print('Copia en Drive', workflow.save_run(PROJECT, SOURCE))

run('-m', 'pip', 'install', '-r', 'requirements.txt')
run('src/preparar_escenas.py', '--desde-manifiesto')


## Entrenar y evaluar
Primero se entrena ObjectCNN con el conjunto interno. Después se ajusta con TACO y se recalculan los umbrales. Los dos scripts seleccionan la época con validación y registran las métricas de prueba al final. El código actual entrena en CPU.


In [ ]:
run('src/objetos.py')
guardar_avance()


In [ ]:
run('src/escenas.py')
guardar_avance()


## Exportar y comprobar el prototipo
El HTML que produce esta celda trae los pesos de la ejecución actual. La comprobación de equivalencia requiere Node.js. La carpeta de Drive contiene el HTML, los pesos, las métricas y los archivos de código de esta ejecución.


In [ ]:
run('src/export_web.py')
if shutil.which('node'):
    run('tests/verificar_html.py', '--guardar')
else:
    print('Node.js no está disponible. Ejecuta la comprobación local indicada en el README.')
guardar_avance()
report = json.loads((PROJECT/'results/escenas.json').read_text(encoding='utf-8'))
print('Exactitud interna', report['test']['accuracy'])
print('F1 macro interno', report['test']['macro_f1'])
print('Prueba externa', report['external_test_after'])
